# Orientation: Mahony Filter using Quaternions

The complementary filter employed earlier was insufficient for the project needs. Therefore, here the Mahony filter is tested out to try to remedy its shortcomings.

## Rotations in 3D space: quaternions

Quaternions are an extension of the complex numbers that live in the four-dimensional space. They are useful because they can be used to represent rotations in 3D space, while, at the same time, avoiding the gimbal lock problem associated with Euler angles. They can be represented as:
$$
\bm{q} = (q_0, q_1, q_2, q_3) = q_0 + q_1 \bm{i} + q_2 \bm{j} + q_3 \bm{k} = (s, \bm{v})
$$
where, in the last notation, $s = q_0$ is the "scalar part" and $\bm{v} = ( q_1, q_2, q_3 )$ is the "vector part". 

Quaternions support addition, subtraction, multiplication, and division, similar to complex numbers. Addition and subtraction work on the components individually. Division is not necessary for this application. For multiplication, the associative property as well as the following identities can be used:
$$
\bm{i}^2 = \bm{j}^2 = \bm{k}^2 = -1 \\
\bm{i} \bm{j} = - \bm{j} \bm{i} = \bm{k} \\
\bm{j} \bm{k} = - \bm{k} \bm{j} = \bm{i}, \\
\bm{k} \bm{i} = - \bm{i} \bm{k} = \bm{j} \\
$$
Note inmediately from these identities that the product is not commutative. Let $\bm{p} = (w, \bm{u})$ be another quaternion. A formula in matrix form for the product can be derived:
$$
\bm{q} \otimes \bm{p} =
\begin{bmatrix}
q_0 & -q_1 & -q_2 & -q_3 \\
q_1 & q_0 & -q_3 & q_2 \\
q_2 & q_3 & q_0 & -q_1 \\
q_3 & -q_2 & q_1 & q_0
\end{bmatrix}
\begin{bmatrix}
p_0 \\
p_1 \\
p_2 \\
p_3
\end{bmatrix} =
\begin{bmatrix}
s w - \bm{v} \cdot \bm{u} \\
s \bm{u} + w \bm{v} + \bm{v} \times \bm{u} \\
\end{bmatrix} 
\ne \bm{p} \otimes \bm{q}
$$

The norm of a quaternion $\bm{q}$ is simply the Euclidean norm of its components:
$$
\|\bm{q}\| = \sqrt{q_0^2 + q_1^2 + q_2^2 + q_3^2}
$$
It can be used to normalize a quaternion:
$$
\hat{\bm{q}} = \frac{\bm{q}}{\|\bm{q}\|}
$$

The conjugate of a quaternion $\bm{q}$ is obtained by negating its vector part:
$$
\bm{q}^* = (q_0, -q_1, -q_2, -q_3)
$$

To rotate a vector $\bm{u} \in \mathbb{R}^3$ using a unit quaternion $\hat{\bm{q}}$, first form a pure quaternion $\bm{p} = (0, \bm{u})$, then apply the rotation as follows:
$$
\bm{p}' = (0, \bm{u}') = \hat{\bm{q}} \otimes \bm{p} \otimes \hat{\bm{q}}^*
$$
Here $\hat{\bm{q}}$ can be expressed as:
$$
\hat{\bm{q}} = \cos(\frac{\beta}{2}) + \sin(\frac{\beta}{2}) (n_1 \bm{i} + n_2 \bm{j} + n_3 \bm{k})
$$
Using this notation, $\hat{\bm{n}} = (n_1, n_2, n_3) \in \mathbb{R}^3$ represents the axis of rotation, and rotation is performed by an angle $\beta$ around this axis, or equivalently, on the plane perpendicular to this axis. This can easily be seen interactively in the source [2]. Alternatively, using vector algebra relations [5] and half-angle trigonometric formulas [6], the quaternion that rotates a vector $\bm{u}$ to the direction of a vector $\bm{u'}$ using the shortest arc possible is:
$$
\hat{\bm{q}}
= (\cos(\frac{\beta}{2}), \sin(\frac{\beta}{2}) \hat{\bm{n}})
= \left(\sqrt{\frac{1 + \cos(\beta)}{2}}, \sqrt{\frac{1 - \cos(\beta)}{2}} \hat{\bm{n}} \right)
= \left(\sqrt{\frac{1 + \hat{\bm{u}} \cdot \hat{\bm{u'}}}{2}}, \sqrt{\frac{1 - \hat{\bm{u}} \cdot \hat{\bm{u'}}}{2}} \frac{\hat{\bm{u}} \times \hat{\bm{u'}}}{\|{\hat{\bm{u}} \times \hat{\bm{u'}}}\|} \right)
$$
Note that this formula assumes that $\bm{u}$ and $\bm{u'}$ are not collinear (i.e. $\bm{u} \times \bm{u'} \neq \bm{0}$). If they are collinear, the rotation axis is not uniquely defined. If they are parallel (i.e. $\bm{u} = k \bm{u'}$, where $k \in \mathbb{R}^+$, or equivalently $\hat{\bm{u}} = \hat{\bm{u'}}$), use $\hat{\bm{n}} = \bm{0}$, since no rotation is needed. If they are antiparallel (i.e. $\bm{u} = -k \bm{u'}$, or equivalently $\hat{\bm{u}} = -\hat{\bm{u'}}$), a $180^\circ$ rotation is needed, so use the basis vector least aligned with $\bm{u}$ as $\hat{\bm{n}}$.

Quaternions are hard visualize on a time series plot directly, but they can be converted to Euler angles with the following formulas [3]:
$$
\phi = atan2(2(q_0 q_1 + q_2 q_3), 1 - 2(q_1^2 + q_2^2)) \\
\theta = asin(2(q_0 q_2 - q_1 q_3)) \\
\psi = atan2(2(q_0 q_3 + q_1 q_2), 1 - 2(q_2^2 + q_3^2))
$$
To go the opposite way, use:
$$
q_0 = \cos\left(\frac{\phi}{2}\right) \cos\left(\frac{\theta}{2}\right) \cos\left(\frac{\psi}{2}\right) + \sin\left(\frac{\phi}{2}\right) \sin\left(\frac{\theta}{2}\right) \sin\left(\frac{\psi}{2}\right) \\
q_1 = \sin\left(\frac{\phi}{2}\right) \cos\left(\frac{\theta}{2}\right) \cos\left(\frac{\psi}{2}\right) - \cos\left(\frac{\phi}{2}\right) \sin\left(\frac{\theta}{2}\right) \sin\left(\frac{\psi}{2}\right) \\
q_2 = \cos\left(\frac{\phi}{2}\right) \sin\left(\frac{\theta}{2}\right) \cos\left(\frac{\psi}{2}\right) + \sin\left(\frac{\phi}{2}\right) \cos\left(\frac{\theta}{2}\right) \sin\left(\frac{\psi}{2}\right) \\
q_3 = \cos\left(\frac{\phi}{2}\right) \cos\left(\frac{\theta}{2}\right) \sin\left(\frac{\psi}{2}\right) - \sin\left(\frac{\phi}{2}\right) \sin\left(\frac{\theta}{2}\right) \cos\left(\frac{\psi}{2}\right)

$$

Sources:
1. [Wikipedia: Quaternion](https://en.wikipedia.org/wiki/Quaternion)
2. [Website: Visualizing Quaternions - Ben Eater & Grant Sanderson (3Blue1Brown)](https://eater.net/quaternions)
3. [Principles of GNSS, Inertial, and Multisensor Integrated Navigation Systems - Paul D. Groves - Chapter 2](../../docs/misc/Principles%20of%20GNSS,%20Inertial,%20and%20Multisensor%20Integrated%20Navigation%20Systems%20-%20Paul%20Groves.pdf)
4. [YouTube: Visualizing the 4d numbers Quaternions](https://www.youtube.com/watch?v=d4EgbgTm0Bg)
5. [Vector Algebra relations: Angles](https://en.wikipedia.org/wiki/Vector_algebra_relations#Angles)
6. [List of trigonometric identities: Half-angle formulas](https://en.wikipedia.org/wiki/List_of_trigonometric_identities#Half-angle_formulas)

## Sensor fusion: Mahony filter

The Mahony filter is a more advanced algorithm that views sensor fusion as a closed loop control system:

* *Error*: angular velocity error derived from accelerometer readings and the true gravity vector.
* *Set point*: constant and equal to zero.
* *Controller*: linear, proportional-integral (PI).
* *Feedback*: instantaneous angular velocity and acceleration readings from the IMU.
* *Physical system*: athlete translating and rotating in space.
* *Output*: corrected angular velocity, ideally approaches zero when perfectly stationary.

In the original paper, orientation was represented using a "Special Orthogonal group (SO(3))", but in the following implementation quaternions will be used.

The steps of a single iteration of the algorithm are as follows:

1. *Estimate current orientation*: making the parallel with the Euler kinematical equations, it can be shown [6] that the quaternion kinematical equation is given by:
   $$
   \dot{\bm{q}}_{w,i}^b = \frac{1}{2} \bm{q}_{w,i-1}^b \otimes (0, \omega_{x,i}, \omega_{y,i}, \omega_{z,i})
   $$
   This can be numerically integrated to yield the current orientation. Using the trapezoidal rule, the integral can be approximated as:
   $$
   \bm{q}_{w,i}^b = \bm{q}_{w,i-1}^b + \frac{\Delta t}{2} (\dot{\bm{q}}_{w,i-1}^b + \dot{\bm{q}}_{w,i}^b)
   $$
   And it also satisfies the property of quaternion norm preservation:
   $$
   \frac{d}{dt} ||\bm{q}_{w,i}^b||^2 = 0
   $$

2. *Obtain gravity vector*: employing the current orientation estimate, transform the gravity vector from the world frame to the body frame: 
   $$
   \bm{\bm{g}}^{b} = \bm{q}_{w,i}^b \otimes \bm{\bm{g}}^{w} \otimes (\bm{q}_{w,i}^b)^*
   $$

3. *Compute the cost*: employ the cross product between the normalized measured specific force and the normalized estimated gravity vector as a measure of the angular velocity error:
   $$
   \bm{e} = 
   k \left(\hat{\bm{g}}^{b} \times -\hat{\bm{f}}^{b}\right) = 
   -k \left(\hat{\bm{f}}^{b} \times \hat{\bm{g}}^{b}\right)
   $$
   Notice that, when the device is stationary, the measured acceleration must resemble the gravity vector ($k = 1$). However, when the device is in motion, there's the influence of dynamic acceleration. A simple way to accomodate for this is to scale down the error by a factor $k \in [0,1)$.

3. *Correct with PI feedback*: compute the gyro bias:
   $$
   \dot{\bm{b}_i} = K_I \cdot \bm{e} \\
   \bm{b}_i = \bm{b}_{i-1} + \frac{\Delta t}{2} (\dot{\bm{b}}_{i-1} + \dot{\bm{b}}_{i})
   $$
   Then correct the measured angular velocity with the estimated bias:
   $$
   \bm{\omega}_{i} \leftarrow \bm{\omega}_{i} - \bm{b}_i + K_P \cdot \bm{e}
   $$
   
4. Re-assign the orientation performing the same operations as in step 1. Normalize the quaternion to ensure it remains a valid rotation representation, as it can be subjected to numerical errors.

The initial rotation $\bm{q}_{w,0}^b$ and bias $\bm{b}_0$ must be set before running the filter. For the rotation, an average of initial stationary acceleration measurements can be used to build an gravity vector, $\bm{\bm{g}}^{b}_0 = -(\bar{f^b_x}, \bar{f^b_y}, \bar{f^b_z})$, and derive an quaternion with the formula discussed above considering $\bm{u} = \bm{g}^w$ and $\bm{u'} = \bm{g}^{b}_0$. For the bias, an average of initial stationary gyroscope measurements can be used to estimate it, $\bm{b}_0 = (\bar{\omega^b_{x}}, \bar{\omega^b_{y}}, \bar{\omega^b_{z}})$. Processing should not begin until these initial values are properly set.

Regarding the filter parameters, they can be interpreted as follows:
* *Proportional gain* ($K_P$): controls how aggressively the filter corrects tilt error from the accelerometer. Too high and the filter becomes jittery. Too low and the filter responds sluggishly.
* *Integral gain* ($K_I$): controls how quickly the filter estimates or learns the gyro bias. Too high and the bias estimate will chase noise. Too low and the filter will be undercorrect for real gyro bias. 

They must be tuned for optimal performance. Heuristics conventionally used such as the Ziegler-Nichols method [7] can't be used here as there is no controlled physical system to run experiments on, so trial and error tuning paired with simulation is necessary. 

1. [Nonlinear Complementary Filters on the Special Orthogonal Group](../../docs/misc/Nonlinear%20Complementary%20Filters%20on%20the%20Special%20Orthogonal%20Group%20-%20Robert%20Mahony.pdf)
2. [Complementary vs. Mahony vs. EKF: Choosing the Right Attitude Estimator for Your Drone](https://husainlokhandwala.in/2026/08/09/attitude-filter-comparison.html)
3. [IMU Mahony filter explanation](https://medium.com/@k66115704/imu-mahony-filter-explanation-1ae75bf033ab)
4. [Introducción al control de sistemas dinámicos lineales continuos - Teoría de Control - ISI - UTN FRSF](../../docs/misc/9-%20Introducción%20al%20Control%20de%20SDLC.pdf)
5. [Controlador PID - Teoría de Control - ISI - UTN FRSF](../../docs/misc/10-%20Controladores%20P+I+D.pdf)
6. [[IONLAB Lectures] Quaternion Kinematics](https://www.youtube.com/watch?v=CecyVl9iXKM)
7. [Wikipedia: Ziegler-Nichols method](https://en.wikipedia.org/wiki/Ziegler%E2%80%93Nichols_method)

## Setup

In [28]:
from signal_utils import (
    IMUSampleReader,
    IMUSampleWriter,
    IMUStationaryChecker,
)
import matplotlib.pyplot as plt
import numpy as np
import math

SAMPLING_FREQUENCY = 30  # Hz
dt = 1 / SAMPLING_FREQUENCY
G = 9.80665  # m/s2
G_VECTOR_W = np.array([0.0, 0.0, -G], dtype=np.float32)

sixFaceStationaryCaptures = [
    "../2-noiseReduction/output/pos-x-up-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
    "../2-noiseReduction/output/neg-x-up-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
    "../2-noiseReduction/output/pos-y-up-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
    "../2-noiseReduction/output/neg-y-up-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
    "../2-noiseReduction/output/pos-z-up-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
    "../2-noiseReduction/output/neg-z-up-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
]
tiltedStationaryCaptures = [
    "../2-noiseReduction/output/pos-x-pos-z-tilt-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
    "../2-noiseReduction/output/neg-x-pos-z-tilt-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
    "../2-noiseReduction/output/pos-y-pos-z-tilt-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
    "../2-noiseReduction/output/neg-y-pos-z-tilt-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
    "../2-noiseReduction/output/pos-x-neg-z-tilt-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
    "../2-noiseReduction/output/neg-x-neg-z-tilt-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
    "../2-noiseReduction/output/pos-y-neg-z-tilt-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
    "../2-noiseReduction/output/neg-y-neg-z-tilt-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
]
stationaryCaptures = [*sixFaceStationaryCaptures, *tiltedStationaryCaptures]

simpleRotationsCaptures = [
    "../2-noiseReduction/output/rot-x-hw-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
    "../2-noiseReduction/output/rot-y-hw-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
    "../2-noiseReduction/output/rot-z-hw-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
]

realExerciseCaptures = [
    "../2-noiseReduction/output/dips-1-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
    "../2-noiseReduction/output/dips-2-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
    "../2-noiseReduction/output/dips-3-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
    "../2-noiseReduction/output/pull-ups-1-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
    "../2-noiseReduction/output/pull-ups-2-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
    "../2-noiseReduction/output/pull-ups-3-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
    "../2-noiseReduction/output/90-deg-push-ups-1-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
    "../2-noiseReduction/output/90-deg-push-ups-2-calib-affine-sixf-tilted-a-online-w-filt-box5.csv",
]

reader = IMUSampleReader()
writer = IMUSampleWriter()
stationaryChecker = IMUStationaryChecker()

## Implement *Quaternion* class

There's available a numpy extension called [numpy-quaternion](https://pypi.org/project/numpy-quaternion/) as well as support from scipy with [scipy.spatial.transform.Rotation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.transform.Rotation.html), but a minimal custom implementation is preferred to improve understanding and provide guidance for the firmware implementation, where no additional dependencies will be used to reduce code space. The class follows the formulas outlined above, works with inmutable objects (safer), overloads operators for convenience, and uses *float32* instead of *float64* to anticipate for precision problems in the device.

In [29]:
class Quaternion:
    def __init__(self, s: float = 0.0, v: np.ndarray = np.zeros(3, dtype=np.float32)):
        self.s = s
        self.v = np.array(v, dtype=np.float32)

    def __str__(self):
        return f"Quaternion(s={self.s}, v={self.v})"

    # Formula: |self - other| < absTol + relTol * |other|
    # Absolute tolerance: fixed, magnitude-independent, useful for values near 0
    # Relative tolerance: scales proportionally with the magnitude, useful for big numbers
    def __eq__(self, other, absTol=1e-8, relTol=1e-5):
        return math.isclose(
            self.s, other.s, abs_tol=absTol, rel_tol=relTol
        ) and np.allclose(self.v, other.v, atol=absTol, rtol=relTol)

    def __add__(self, other):
        if not isinstance(other, Quaternion):
            return NotImplemented
        return Quaternion(
            self.s + other.s,
            self.v + other.v,
        )

    def __mul__(self, other):
        if isinstance(other, Quaternion):
            return Quaternion(
                self.s * other.s - np.dot(self.v, other.v),
                self.s * other.v + other.s * self.v + np.cross(self.v, other.v),
            )
        if isinstance(other, (int, float)):
            return Quaternion(
                self.s * other,
                self.v * other,
            )
        return NotImplemented

    # Scalar multiplication from the left: forward to __mul__() commuting the operators
    def __rmul__(self, scalar):
        return self * scalar

    def conjugate(self):
        return Quaternion(self.s, -self.v)

    def norm(self):
        return math.sqrt(self.s**2 + np.dot(self.v, self.v))

    def normalized(self):
        magnitude = self.norm()
        if math.isclose(magnitude, 0.0):
            raise ValueError("Cannot normalize a zero quaternion.")
        return Quaternion(self.s / magnitude, self.v / magnitude)

    def rotate(self, u):
        magnitude = self.norm()
        if math.isclose(magnitude, 0.0):
            raise ValueError("Cannot rotate a vector with a zero quaternion.")
        if not math.isclose(magnitude, 1.0):
            raise ValueError("Quaternion must be normalized to rotate a vector.")
        p = Quaternion(0.0, np.array(u, dtype=np.float32))
        rotated = self * p * self.conjugate()
        return rotated.v

    def toEulerAngles(self, toDegrees=True):
        roll = math.atan2(
            2.0 * (self.s * self.v[0] + self.v[1] * self.v[2]),
            1.0 - 2.0 * (self.v[0] ** 2 + self.v[1] ** 2),
        )
        pitch = math.asin(2 * (self.s * self.v[1] - self.v[0] * self.v[2]))
        yaw = math.atan2(
            2.0 * (self.s * self.v[2] + self.v[0] * self.v[1]),
            1.0 - 2.0 * (self.v[1] ** 2 + self.v[2] ** 2),
        )

        if toDegrees:
            roll = math.degrees(roll)
            pitch = math.degrees(pitch)
            yaw = math.degrees(yaw)

        return np.array(
            [
                roll,
                pitch,
                yaw,
            ]
        )

    @staticmethod
    def fromEulerAngles(roll, pitch, yaw, inDegrees=True):
        rollRad = roll
        pitchRad = pitch
        yawRad = yaw
        if inDegrees:
            rollRad = math.radians(roll)
            pitchRad = math.radians(pitch)
            yawRad = math.radians(yaw)

        cosRoll = math.cos(0.5 * rollRad)
        sinRoll = math.sin(0.5 * rollRad)
        cosPitch = math.cos(0.5 * pitchRad)
        sinPitch = math.sin(0.5 * pitchRad)
        cosYaw = math.cos(0.5 * yawRad)
        sinYaw = math.sin(0.5 * yawRad)

        s = cosRoll * cosPitch * cosYaw + sinRoll * sinPitch * sinYaw
        v = np.zeros(3, dtype=np.float32)
        v[0] = sinRoll * cosPitch * cosYaw - cosRoll * sinPitch * sinYaw
        v[1] = cosRoll * sinPitch * cosYaw + sinRoll * cosPitch * sinYaw
        v[2] = cosRoll * cosPitch * sinYaw - sinRoll * sinPitch * cosYaw

        return Quaternion(s, v)


def findQuaternionFromVectors(ui: np.ndarray, uf: np.ndarray) -> Quaternion:
    uiHat = ui / np.linalg.norm(ui)
    ufHat = uf / np.linalg.norm(uf)
    n = np.cross(uiHat, ufHat)
    if np.allclose(n, np.zeros(3)):
        if np.allclose(uiHat, ufHat):
            nHat = np.zeros(3, dtype=np.float32)
        else:
            # Find basis vector with least contribution in the initial vector,
            # and get it conveniently from the identity matrix
            nHat = np.eye(3)[:, np.argmin(np.abs(uiHat))]
    else:
        nHat = n / np.linalg.norm(n)

    # Normalization is needed for the collinear cases
    return Quaternion(
        s=math.sqrt((1 + np.dot(uiHat, ufHat)) / 2),
        v=math.sqrt((1 - np.dot(uiHat, ufHat)) / 2) * nHat,
    ).normalized()


def findQuaternionFromAngleAndNormalVector(
    angle: float, n: np.ndarray, inDegrees=True
) -> Quaternion:
    angleRad = angle
    if inDegrees:
        angleRad = math.radians(angle)
    return Quaternion(
        s=math.cos(angleRad / 2),
        v=math.sin(angleRad / 2) * n / np.linalg.norm(n),
    )

In [30]:
# Rotation by the identity quaternion
ui = np.array([1, 0, 0], dtype=np.float32)
q = Quaternion(1, np.array([0.0, 0.0, 0.0]))
uf = q.rotate(ui)
np.testing.assert_allclose(ui, uf)

# Rotation given angle and normal vector
ui = np.array([1.0, 0.0, 0.0], dtype=np.float32)
q = findQuaternionFromAngleAndNormalVector(
    angle=90.0,
    n=np.array([0, 0, 1.0], dtype=np.float32),
    inDegrees=True,
)
uf = q.rotate(ui)
np.testing.assert_allclose(uf, np.array([0.0, 1.0, 0.0], dtype=np.float32))
assert math.isclose(np.linalg.norm(uf), 1.0, abs_tol=1e-5)

# Rotation given not collinear initial and final vectors
ui = np.array([1.0, 0.0, 0.0], dtype=np.float32)
uf = np.array([0.0, 1.0, 0.0], dtype=np.float32)
q = findQuaternionFromVectors(ui, uf)
np.testing.assert_allclose(q.rotate(ui), uf)

# Rotation given parallel initial and final vectors
ui = np.array([1.0, 0.0, 0.0], dtype=np.float32)
uf = 2 * ui
q = findQuaternionFromVectors(ui, uf)
uf = q.rotate(ui)
np.testing.assert_allclose(ui, uf)

# Rotation given antiparallel initial and final vectors
ui = np.array([1.0, 0.0, 0.0], dtype=np.float32)
uf = -2 * ui
q = findQuaternionFromVectors(ui, uf)
uf = q.rotate(ui)
np.testing.assert_allclose(uf, -ui)

# Conversion from and to Euler angles
angles = np.array([30.0, 60.0, 45.0], dtype=np.float32)
np.testing.assert_allclose(
    Quaternion.fromEulerAngles(*angles).toEulerAngles(), angles, rtol=1e-5, atol=1e-5
)

## Implement Mahony filter

In [ ]:
def mahonyFilter(a, w, dt, KM, KP, KI):
    s = np.zeros(len(a), dtype=np.float32)
    v = np.zeros((len(a), 3), dtype=np.float32)
    b = np.zeros(len(a), dtype=np.float32)
    e = np.zeros((len(a), 3), dtype=np.float32)

    q0 = findQuaternionFromVectors(G_VECTOR_W, a[0, :])
    s[0], v[0] = q0.s, q0.v
    prevQ = q0
    prevQRate = Quaternion()

    b0 = 0
    b[0] = b0
    prevBRate = b0

    for i in range(1, len(a)):
        qRate = 1 / 2 * prevQ * Quaternion(0.0, w[i])
        q = prevQ + dt / 2 * (prevQRate + qRate)
        q = q.normalized()

        gB = q.rotate(G_VECTOR_W)

        e[i] = -np.cross(a[i] / np.linalg.norm(a), gB / np.linalg.norm(gB))
        if not stationaryChecker.isStationarySample(*a[i, :], *w[i, :]):
            e[i] *= KM

        bRate = KI * e[i]
        b[i] = b[i - 1] + dt / 2 * (prevBRate + bRate)

        w[i] = w[i] + b[i] + KP * e[i]

        qRate = 1 / 2 * prevQ * Quaternion(0.0, w[i])
        q = prevQ + dt / 2 * (prevQRate + qRate)
        q = q.normalized()

        prevBRate = bRate
        s[i], v[i] = q.s, q.v
        prevQRate = qRate
        prevQ = q

    return s, v, b, e